In [1]:
%%capture
!pip install aixplain

In [2]:
import os
os.environ["AIXPLAIN_API_KEY"] = "<YOUR_API_KEY>"

from aixplain.factories import AgentFactory, ToolFactory, ModelFactory

In [3]:
# Get required tools
scraper_tool = ToolFactory.get("66f423426eb563fa213a3531")  # Scrape2
serp_tool = ToolFactory.get("6931bdf462eb386b7158def3")  # SERP Scraper

In [4]:
# Get Claude Code model
claude_code = ModelFactory.get("<CLAUDE_MODEL_ID>")  # Replace with actual Claude Code model ID

In [5]:
# Create Backlink Prospector Agent
backlink_agent = AgentFactory.create(
    name="Backlink Prospector",
    description="An SEO assistant that finds and qualifies backlink prospects",
    llm_id=claude_code.id,
    instructions="""
    You are an SEO assistant specialized in finding backlink opportunities.

    PROCESS:
    1. Use the SERP scraper tool to find 5-10 relevant websites about the given topic/keyword
    2. For each website found:
       a. Use the scraper tool to fetch and extract text from the homepage
       b. Search the extracted text for email addresses (look for patterns like name@domain.com)
       c. If no email found on homepage, check if there's a contact page URL
       d. If contact page exists, scrape it and look for emails
       e. Note if only a contact form is found (no direct email)
       f. If no email and no contact form, move to next website

    3. Evaluate each prospect:
       - Is the site relevant to the topic?
       - Does it have contact information?
       - Is it a quality backlink opportunity?

    4. At the end, output an array of all valid prospects with this structure:
       [
         {
           "site_url": "https://example.com",
           "email_list": ["contact@example.com", "info@example.com"],
           "contact_page": "https://example.com/contact",
           "has_contact_form": "yes/no",
           "notes": "Brief notes about the site and why it's a good prospect"
         }
       ]

    IMPORTANT:
    - Only include sites where you found at least an email OR a contact form
    - Be thorough in checking for emails (they may be in footer, contact page, about page)
    - Format the final output as a clean JSON array
    - Skip sites that have no way to contact them
    """,
    tools=[
        AgentFactory.create_model_tool(
            model=serp_tool.id,
            description="Search Google for websites related to a topic. Input should be a search query."
        ),
        AgentFactory.create_model_tool(
            model=scraper_tool.id,
            description="Scrape website content from a URL. Input must be a valid URL."
        )
    ]
)

print(f"Backlink Prospector Agent created! ID: {backlink_agent.id}")




/tmp/ipython-input-2461082095.py:2: DeprecationWarning: The 'llm_id' parameter is deprecated and will be removed in a future version. Use the 'llm' parameter instead by passing the LLM ID or LLM instance directly.
  backlink_agent = AgentFactory.create(
/usr/local/lib/python3.12/dist-packages/aixplain/factories/agent_factory/__init__.py:146: UserWarning: Deprecating 'llm_id', use `llm` to define the large language model in agents.
  warnings.warn(


Backlink Prospector Agent created! ID: 695d3ab2a87cf6b5e1aff927
🤖  Backlink Prospector | ⏳ Initialization
⚙️  Backlink Prospector | Claude 3.7 Sonnet | ⏳
⚙️  Backlink Prospector | search | ⏳
⚙️  Backlink Prospector | Claude 3.7 Sonnet | ⏳
⚙️  Backlink Prospector | utilities-aixplain-scrape_website_tool | ⏳
⚙️  Backlink Prospector | Claude 3.7 Sonnet | ⏳
⚙️  Backlink Prospector | utilities-aixplain-scrape_website_tool | ⏳
⚙️  Backlink Prospector | Claude 3.7 Sonnet | ⏳
⚙️  Backlink Prospector | utilities-aixplain-scrape_website_tool | ✓ The output of the function 'utilities' to the input '{'text': 'https://www.curiouslyconscious.com//contact'}' is: Page not found | Curiously Conscious
About
Beauty
Fashion
Food
Lifestyle
Travel
Nothin...
✅ Done | (0.0 s total | $0)

BACKLINK PROSPECTS FOUND:
Error Code: AX-INT-1000 - ```json
{
  "error_message": "Operation Timed Out - The backlink prospect search reached its iteration limit. The agent was attempting to find contact information for sustai

In [6]:
# Example usage
response = backlink_agent.run(
    query="""
    Find backlink prospects for the topic: "sustainable fashion brands"

    Look for fashion blogs, lifestyle websites, or sustainability-focused sites
    that might be interested in linking to sustainable fashion content.
    """,
    max_iterations=50
)

print("\n" + "="*50)
print("BACKLINK PROSPECTS FOUND:")
print("="*50)
print(response.data.output)

🤖  Backlink Prospector | ⏳ Initialization
⚙️  Backlink Prospector | Claude 3.7 Sonnet | ⏳
⚙️  Backlink Prospector | search | ⏳
⚙️  Backlink Prospector | Claude 3.7 Sonnet | ⏳
⚙️  Backlink Prospector | utilities-aixplain-scrape_website_tool | ✓ The output of the function 'utilities' to the input '{'text': 'https://www.curiouslyconscious.com//contact'}' is: Page not found | Curiously Conscious
About
Beauty
Fashion
Food
Lifestyle
Travel
Nothin...
⚙️  Backlink Prospector | utilities-aixplain-scrape_website_tool | ✓ The output of the function 'utilities' to the input '{'text': 'https://www.curiouslyconscious.com//about'}' is: About | Curiously Conscious
About
Beauty
Fashion
Food
Lifestyle
Travel
Hi, I’m Besma
I’m...
⚙️  Backlink Prospector | Claude 3.7 Sonnet | ⏳
⚙️  Backlink Prospector | utilities-aixplain-scrape_website_tool | ⏳
⚙️  Backlink Prospector | Claude 3.7 Sonnet | ⏳
🤖  Backlink Prospector | ⏳ Formatting Output
✅ Done | (144.0 s total | 38 API calls | $1.027209)

BACKLINK PROSPEC

In [7]:
 #Follow-up to refine results
session_id = response.data.session_id

response = backlink_agent.run(
    query="Can you filter the list to only show prospects with direct email addresses (exclude contact form only)?",
    session_id=session_id
)

print("\n" + "="*50)
print("FILTERED RESULTS:")
print("="*50)
print(response.data.output)


🤖  Backlink Prospector | ⏳ Initialization
⚙️  Backlink Prospector | Claude 3.7 Sonnet | ⏳
🤖  Backlink Prospector | ⏳ Formatting Output
✅ Done | (19.8 s total | 2 API calls | $0.029451)

FILTERED RESULTS:
[
  {
    "site_url": "https://ohsevendays.com/blogs/news/10-best-sustainable-fashion-blogs-you-need-to-follow?srsltid=AfmBOoq7U8Plh5uQi5zl927R8vpRV1T3GgDg5cS-9Ib8hvsPaDzFJ_y2",
    "email_list": ["info@ohsevendays.com"],
    "contact_page": "https://ohsevendays.com/pages/contact-us",
    "has_contact_form": "yes",
    "notes": "OhSevenDays is a sustainable fashion brand with its own blog that features content about sustainable fashion practices. They already write about other sustainable fashion blogs, making them a good prospect for backlinks. Their site is focused on upcycling and slow fashion."
  },
  {
    "site_url": "https://www.curiouslyconscious.com/",
    "email_list": [],
    "contact_page": "https://www.curiouslyconscious.com//about",
    "has_contact_form": "yes",
    "not

In [ ]:
# Deploy the agent
backlink_agent.deploy()

print(f"\nBacklink Prospector deployed!")
print(f"Access at: https://platform.aixplain.com/discover/agent/{backlink_agent.id}")